# The oracle gate under four deadline settings

Reproduces the tables in the status report from the raw simulator runs in `data/freeinference/gate*/`.
Everything here is computed straight from the `.jsonl` result files (one line per simulator run), not from
`analysis/replay/report.py`; the last cell cross-checks the two.

| Section | What it confirms in the status report |
|---|---|
| 1 | the files are complete: run counts, no duplicates, the deadline settings |
| 2 | the table of attainment by deadline setting and memory pool |
| 3 | the same numbers against what the status report prints |
| 4 | the findings row: max, median and cells at 3+ and 5+ points, and the verdict |
| 5 | where the session-deadline gain sits (best cells, and by load) |
| 6 | the latency rule (shortest-remaining-time against attained service) |
| 7 | a cross-check against `report.py` |

The *future* step is what a perfect view of the future adds over EDF, the only step a predictor can supply.
It uses only the standard library (the last cell also imports the repo's `report.py`). Run the cells top to bottom.

## 1. Load the runs

In [ ]:
import json
import statistics as st
from collections import defaultdict
from pathlib import Path

# find the repo root from wherever the notebook is opened
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "analysis" / "replay").is_dir())
DATA = ROOT / "data" / "freeinference"

# deadline setting -> result file. "Turn" = one deadline per user turn; "Session" = one per whole session
# (user pauses, 5 s here, count inside it). "class" = a random class budget; "proportional" = slack x the
# unit's time alone on the server.
SETTINGS = {
    "Turn, class":           DATA / "gate" / "class_l40s.jsonl",
    "Turn, proportional":    DATA / "gate" / "prop_l40s.jsonl",
    "Session, class":        DATA / "gate_session_reply5" / "class_l40s.jsonl",
    "Session, proportional": DATA / "gate_session_reply5" / "prop_l40s.jsonl",
}
REALISTIC_KV = (1e6, 2e6, 4e6)   # KV-cache tokens: roughly one to two large GPUs
SMALL, LARGE = 3.0, 5.0          # points of attainment: below SMALL drop, from LARGE build (fixed before the first run)
JCT_BAR = 10.0                   # percent latency reduction against attained service that would justify a predictor
LAS, EDF = ("las", "lru"), ("edf", "lru")
ORACLES = [("llf_oracle", "lru"), ("srpt_oracle", "lru")]

RAW = {name: [json.loads(line) for line in open(path)] for name, path in SETTINGS.items()}
KEY = ("profile", "day", "load", "kv", "mode", "tight", "policy", "eviction", "slots")
for name, rows in RAW.items():
    keys = [tuple(r[k] for k in KEY) for r in rows]
    print(f"{name:22} {len(rows):6,} runs | {'no duplicate runs' if len(set(keys)) == len(keys) else 'DUPLICATE RUNS'} | "
          f"{len({r['day'] for r in rows})} days | {len({(r['policy'], r['eviction']) for r in rows})} policy pairs | "
          f"mode {sorted({r['mode'] for r in rows})} | tightness/slack {sorted({r['tight'] for r in rows})}")

## 2. Attainment by deadline setting and memory pool

Per (load, tightness or slack, KV) cell, the mean over the test days of:
**LAS** (attained service, the ATLAS stand-in), **EDF** (adds the deadlines) and the **oracle** (adds the future;
the better of oracle-LLF and oracle-SRPT, as the report picks it). The table is the mean over the 9 (class) or
12 (proportional) cells at each memory size.

In [ ]:
def cells(rows, field="attainment"):
    """(load, tight, kv) -> (policy, eviction) -> {day: value}"""
    out = defaultdict(lambda: defaultdict(dict))
    for r in rows:
        if r.get(field) is not None:
            out[(r["load"], r["tight"], r["kv"])][(r["policy"], r["eviction"])][r["day"]] = r[field]
    return out


def day_mean(cell, pair):
    v = cell.get(pair)
    return st.mean(v.values()) if v else None


def ladder(rows):
    """One entry per realistic-KV cell: attainment (%) at each rung and the two steps in points."""
    out = []
    for (load, tight, kv), c in sorted(cells(rows).items()):
        if kv not in REALISTIC_KV:
            continue
        las, edf = day_mean(c, LAS), day_mean(c, EDF)
        scored = [(day_mean(c, p), p) for p in ORACLES if day_mean(c, p) is not None]
        if las is None or edf is None or not scored:
            continue
        orc, pol = max(scored)
        out.append(dict(load=load, tight=tight, kv=kv, las=100 * las, edf=100 * edf, oracle=100 * orc,
                        oracle_policy=pol[0], deadlines=100 * (edf - las), future=100 * (orc - edf)))
    return out


LADDER = {name: ladder(rows) for name, rows in RAW.items()}

TABLE = []   # the status report's table: (setting, KV in M, cells, LAS, EDF, oracle, deadlines step, future step)
print(f"{'Deadline setting':22} {'KV':>3} {'cells':>5} | {'LAS':>6} {'EDF':>6} {'Oracle':>7} | {'EDF-LAS':>8} {'oracle-EDF':>11}")
for name, rows in LADDER.items():
    for kv in REALISTIC_KV:
        v = [r for r in rows if r["kv"] == kv]
        m = lambda k: st.mean(r[k] for r in v)
        TABLE.append((name, int(kv / 1e6), len(v), m("las"), m("edf"), m("oracle"), m("deadlines"), m("future")))
        print(f"{name:22} {int(kv / 1e6):>2}M {len(v):>5} | {m('las'):5.1f}% {m('edf'):5.1f}% {m('oracle'):6.1f}% | "
              f"{m('deadlines'):+8.1f} {m('future'):+11.1f}")
    print()

## 3. Compare with the numbers in the status report

The values below were copied from the table in the status report (rev 83). Every row should say `ok`.

In [ ]:
REPORTED = {   # (setting, KV in M): LAS, EDF, oracle, deadlines step, future step
    ("Turn, class", 1): (87.2, 90.8, 91.2, 3.6, 0.4),            ("Turn, class", 2): (92.9, 94.0, 94.4, 1.2, 0.3),
    ("Turn, class", 4): (97.4, 97.8, 98.0, 0.4, 0.2),
    ("Turn, proportional", 1): (59.6, 72.4, 75.7, 12.7, 3.4),    ("Turn, proportional", 2): (75.0, 78.9, 80.5, 3.9, 1.6),
    ("Turn, proportional", 4): (91.4, 92.9, 93.3, 1.5, 0.4),
    ("Session, class", 1): (78.6, 89.1, 92.7, 10.5, 3.7),        ("Session, class", 2): (92.6, 95.5, 96.1, 2.9, 0.6),
    ("Session, class", 4): (97.4, 98.7, 98.8, 1.2, 0.1),
    ("Session, proportional", 1): (54.7, 60.5, 71.8, 5.8, 11.3), ("Session, proportional", 2): (80.5, 85.6, 87.6, 5.1, 2.0),
    ("Session, proportional", 4): (95.2, 97.4, 97.9, 2.2, 0.5),
}
bad = 0
for name, kv, n, *vals in TABLE:
    diff = [round(v, 1) - r for v, r in zip(vals, REPORTED[(name, kv)])]
    ok = all(abs(d) < 0.051 for d in diff)
    bad += not ok
    print(f"{name:22} {kv}M  {'ok' if ok else 'DIFFERS ' + str([round(d, 1) for d in diff])}")
print("\nall 12 rows match the status report" if not bad else f"\n{bad} rows differ")

## 4. The findings row: the future step, cell by cell

The verdict rule was fixed before the first run, on the largest gain in any realistic cell: under 3 points
drop the predictor, 5 or more build, in between noise-test. Note that it reads the **maximum**, so look at the
median and the by-memory means beside it.

In [ ]:
print(f"{'Deadline setting':22} {'max':>6} {'median':>7} {'cells 3+':>9} {'cells 5+':>9} | {'mean at 1M':>10} {'2M':>6} {'4M':>6} | verdict")
for name, rows in LADDER.items():
    g = [r["future"] for r in rows]
    verdict = "drop" if max(g) < SMALL else ("build" if max(g) >= LARGE else "noise-test")
    by_kv = [st.mean(r["future"] for r in rows if r["kv"] == kv) for kv in REALISTIC_KV]
    print(f"{name:22} {max(g):+6.1f} {st.median(g):+7.1f} {sum(x >= SMALL for x in g):>4}/{len(g):<4} {sum(x >= LARGE for x in g):>4}/{len(g):<4} | "
          f"{by_kv[0]:+10.1f} {by_kv[1]:+6.1f} {by_kv[2]:+6.1f} | {verdict}")

## 5. Where the session-deadline gain sits

The status report says the gain is concentrated in the memory-starved corner. Check which cells carry it.

In [ ]:
for name in ("Session, class", "Session, proportional"):
    rows = LADDER[name]
    top = sorted(rows, key=lambda r: -r["future"])[:6]
    print(f"{name}: the six cells with the largest future step")
    for r in top:
        print(f"   +{r['future']:5.1f}  load {r['load']}  {'tightness' if 'class' in name else 'slack'} {r['tight']}  KV {int(r['kv'] / 1e6)}M   "
              f"LAS {r['las']:.1f}%  EDF {r['edf']:.1f}%  oracle {r['oracle']:.1f}%  ({r['oracle_policy']})")
    big = [r for r in rows if r["future"] >= SMALL]
    print(f"   cells at {SMALL:.0f}+ points: {len(big)} of {len(rows)}, by KV: "
          + ", ".join(f"{int(kv / 1e6)}M {sum(r['kv'] == kv for r in big)}" for kv in REALISTIC_KV))
    print("   mean future step by load:   " + ", ".join(
        f"load {ld}: {st.mean(r['future'] for r in rows if r['load'] == ld):+.1f}" for ld in sorted({r['load'] for r in rows})))
    print("   mean future step at 1M KV by load: " + ", ".join(
        f"{ld}: {st.mean(r['future'] for r in rows if r['load'] == ld and r['kv'] == 1e6):+.1f}" for ld in sorted({r['load'] for r in rows})))
    print()

## 6. The latency rule

Pure shortest-remaining-time policies (no deadline logic) against attained service (LAS): the percent drop in
mean and p95 turn latency, per cell, averaged over days, for the history version and the oracle version. The rule
asks for 10% or more. It reads a **maximum** too, so the median is printed beside it. Cells are the realistic memory
sizes plus unlimited memory (4 sizes x 9 cells = 36 for class budgets, 48 for proportional ones, whose latency
numbers repeat the class ones because these policies ignore deadlines).

In [ ]:
def drop(cell, pair):
    """Percent drop of a latency metric against LAS, mean over days (positive = faster)."""
    a, b = cell.get(pair, {}), cell.get(LAS, {})
    days = [d for d in a if d in b and b[d] > 0]
    return st.mean(100 * (b[d] - a[d]) / b[d] for d in days) if days else None


def latency_cells(rows):
    mean_l, p95_l = cells(rows, "mean_latency"), cells(rows, "p95_latency")
    out = []
    for k in sorted(mean_l):
        if k[2] != 0 and k[2] not in REALISTIC_KV:
            continue
        vals = (drop(mean_l[k], ("psrpt_cheap", "lru")), drop(mean_l[k], ("psrpt_oracle", "lru")),
                drop(p95_l.get(k, {}), ("psrpt_cheap", "lru")), drop(p95_l.get(k, {}), ("psrpt_oracle", "lru")))
        if None not in vals:
            out.append((k, vals))
    return out


print(f"{'Deadline setting':22} {'metric':>6} | {'history (typical)':>17} | {'oracle max':>10} {'median':>7} {'cells >= 10%':>13}")
for name, rows in RAW.items():
    lat = latency_cells(rows)
    for label, i_cheap, i_orc in (("mean", 0, 1), ("p95", 2, 3)):
        c, o = [v[i_cheap] for _, v in lat], [v[i_orc] for _, v in lat]
        print(f"{name:22} {label:>6} | {st.median(c):+16.1f}% | {max(o):+9.1f}% {st.median(o):+6.1f}% {sum(x >= JCT_BAR for x in o):>7}/{len(o)}")

## 7. Cross-check against `report.py`

The same ladder from the report's own code. Differences here would mean this notebook and `report.py` disagree
about what the policies or the cells are.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from analysis.replay import report

for name, path in SETTINGS.items():
    ref = report.ladder_rows(report.load(str(path)))
    mine = LADDER[name]
    same = len(ref) == len(mine) and all(
        all(abs(a[k] - b[k]) < 1e-9 for k in ("las", "edf", "oracle", "deadlines", "future")) for a, b in zip(ref, mine))
    print(f"{name:22} {len(mine):>2} cells  {'matches report.py' if same else 'DIFFERS from report.py'}")

## What this does not check

* **That the sweeps themselves are right.** These are the results as the simulator wrote them. To regenerate:
  `analysis/replay/run_all.sh` (turn-level) and
  `UNIT=session REPLY=fixed:5 OUT=data/freeinference/gate_session_reply5 SWEEPS=class_l40s,prop_l40s analysis/replay/run_all.sh` (session-level).
* **That the deadlines are realistic.** FreeInference has no service-level targets; both budget rules are synthetic.
* **What the session-level oracle knows.** It knows every later turn of the session, including how many messages the
  user will still send. A forecaster of the requests left in the current turn does not predict that, so read the
  session-level "build" as an upper bound for a whole-session predictor.
* **The two weeks only.** Train Aug 23-29, replay Aug 30-Sep 5.